# 05 · Anchor products: what is in the first basket of loyal customers?

**Business question:** are there products in a customer's **first order** that go together with becoming a **weekly** shopper? If so, a grocer like Picnic could feature them in onboarding (welcome offers, first-order suggestions).

**Loyalty measure:** a customer is *weekly* if their median gap between orders is ≤ 8 days (same rule as notebook 03).

**Important:** this is a **correlation**, not proof of cause. Customers who buy bananas in their first order may simply be families who shop weekly anyway.

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("../data/instacart.duckdb", read_only=True)

def q(sql):
    """Run a SQL query and return a pandas DataFrame."""
    return con.sql(sql).df()

## 5.1 Baseline: share of weekly customers
Build a CTE `user_profile` with one row per user: `user_id`, `n_orders`, `median_gap`, `is_weekly` (1 if median_gap ≤ 8, else 0).
What % of customers are weekly? (it should match notebook 03)

## 5.2 What goes into first baskets?
The products bought most often in a customer's **first order** (`order_number = 1`): top 15 by number of customers.

In [3]:
q("""
WITH user_profile AS (
    SELECT user_id,
           COUNT(*) AS n_orders,
           MEDIAN(days_since_prior_order) AS median_gap,
           CASE WHEN MEDIAN(days_since_prior_order) <= 8 THEN 1 ELSE 0 END AS is_weekly
    FROM orders
    GROUP BY user_id
)
SELECT COUNT(*)                        AS n_users,
       ROUND(AVG(is_weekly) * 100, 1)        AS pct_weekly
FROM user_profile
""")

,n_users,pct_weekly
0,206209,31.9


## 5.3 Anchor products
For each product bought in the first order of **at least 1,000 customers**:
- `n_customers`
- `pct_weekly`: % of those customers who became weekly shoppers
- `lift`: `pct_weekly` divided by the baseline from 5.1
- `rank_lift`: rank by lift with `RANK() OVER (...)`

Show the top 15 and the bottom 15.

## 5.4 Fresh share of the first basket
For each customer: % of fresh items in their first order. Split customers into 4 equal groups with `NTILE(4)` and compute `pct_weekly` per group. Save as `fresh_q` with columns `fresh_quartile`, `avg_fresh_share`, `pct_weekly`.

In [ ]:
fresh_q = q("""

""")
fresh_q

### Chart: weekly customers by fresh share of the first basket

In [ ]:
BASELINE = 31.9  # % weekly customers overall (from 5.1) - update if different

labels = [f"Q{int(qq)}\n{s:.0f}% fresh" for qq, s in zip(fresh_q["fresh_quartile"], fresh_q["avg_fresh_share"])]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, fresh_q["pct_weekly"], color="#2a78d6", width=0.6)
for b, v in zip(bars, fresh_q["pct_weekly"]):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.0f}%", ha="center", fontsize=10, color="#444")
ax.axhline(BASELINE, color="#888", linewidth=1, linestyle="--")
ax.text(len(labels) - 0.5, BASELINE + 0.8, f"all customers: {BASELINE:.0f}%", ha="right", fontsize=9, color="#666")
ax.yaxis.set_visible(False)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#ccc")
ax.tick_params(axis="x", length=0, labelsize=9)
ax.set_title("Weekly shoppers by fresh share of the first basket", loc="left", fontsize=12)
plt.tight_layout()
plt.savefig("../figures/fresh_first_basket.png", dpi=150, bbox_inches="tight")
plt.show()

## Findings & recommendation

- 
- 
- 

**Caveat:** correlation, not causation.